# Mass Matrix Validation for Nodal DG Method

Validate discrete mass matrix orthonormality using Dubiner basis.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.core.generators import build_nodes
from src.core.render_utils import VERTICES_2D, VERTICES_3D
from src.bases import vandermonde_2d_dubiner


def get_reference_data(method: str, k: int):
    nodes = build_nodes(method, k, VERTICES_2D, VERTICES_3D)
    bary_coords = np.array([n.barycentric for n in nodes], dtype=float)
    weights = np.array([n.weight for n in nodes], dtype=float)
    xi = 2.0 * bary_coords[:, 2] - 1.0
    eta = 2.0 * bary_coords[:, 1] - 1.0
    return {
        "nodes": nodes,
        "bary_coords": bary_coords,
        "weights": weights,
        "xi": xi,
        "eta": eta,
    }

In [2]:
METHODS = ["table1", "table2"]
K_VALUES = [1, 2, 3, 4]
TOLERANCE_MASS = 1e-13

In [3]:
rows = []
global_passed = True

for method in METHODS:
    for k in K_VALUES:
        ref = get_reference_data(method, k)
        xi = ref["xi"]
        eta = ref["eta"]
        weights = ref["weights"]

        num_basis = (k + 1) * (k + 2) // 2
        V = vandermonde_2d_dubiner(xi, eta, k)
        W = np.diag(weights * 2.0)
        M = V.T @ W @ V

        M_error = M - np.eye(num_basis)
        max_error = float(np.max(np.abs(M_error)))
        frob_norm = float(np.linalg.norm(M_error, ord="fro"))

        identity_pass = max_error < TOLERANCE_MASS

        if identity_pass:
            status = "✓ PASS"
            contributes_failure = False
        elif method == "table1":
            status = "⚠ EXPECTED FAIL"
            contributes_failure = False
        else:
            status = "✗ FAIL"
            contributes_failure = True

        if contributes_failure:
            global_passed = False

        rows.append({
            "method": method,
            "k": k,
            "num_nodes": len(ref["nodes"]),
            "num_basis": num_basis,
            "max_abs_M_minus_I": max_error,
            "fro_norm_M_minus_I": frob_norm,
            "status": status,
            "identity_pass": identity_pass,
            "is_expected_fail": (status == "⚠ EXPECTED FAIL"),
        })

mass_df = pd.DataFrame(rows)
display(mass_df)

,method,k,num_nodes,num_basis,max_abs_M_minus_I,fro_norm_M_minus_I,status,identity_pass,is_expected_fail
0,table1,1,6,3,1.000000e+00,1.414214e+00,⚠ EXPECTED FAIL,False,True
1,table1,2,10,6,9.166667e-01,1.468098e+00,⚠ EXPECTED FAIL,False,True
2,table1,3,18,10,9.305556e-01,1.583049e+00,⚠ EXPECTED FAIL,False,True
3,table1,4,22,15,8.849569e-01,1.624557e+00,⚠ EXPECTED FAIL,False,True
4,table2,1,3,3,6.661338e-16,7.505604e-16,✓ PASS,True,False
5,table2,2,6,6,1.706968e-15,4.238968e-15,✓ PASS,True,False
6,table2,3,12,10,2.553513e-15,3.916818e-15,✓ PASS,True,False
7,table2,4,16,15,2.442491e-15,5.041244e-15,✓ PASS,True,False


In [4]:
summary_df = (
    mass_df.groupby(["method", "status"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
display(summary_df)

if global_passed:
    print("✓ SUCCESS! No unexpected mass-matrix failures were detected.")
else:
    print("✗ FAILURE! At least one non-table1 case failed strict orthonormality.")

,method,status,count
0,table1,⚠ EXPECTED FAIL,4
1,table2,✓ PASS,4


✓ SUCCESS! No unexpected mass-matrix failures were detected.
